<a href="https://colab.research.google.com/github/AIML-Dept/Adhisha_Sreedith_1GA23AI002/blob/main/Week_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.7 MB/s eta 0:00:00


# **1) Superdense Coding: From Basic Circuit to Noisy-Channel Simulation**

In [2]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

# Prepare the shared Bell pair
qc.h(0)
qc.cx(0, 1)

# Alice encodes '00': no operation

# Bob decodes
qc.cx(0, 1)
qc.h(0)

# Measure both qubits
qc.measure([0, 1], [0, 1])

simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()

print("Measurement counts:", counts)
print(qc.draw())


Measurement counts: {'00': 1024}
     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├──■────■──┤ H ├┤M├
     └───┘┌─┴─┐┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├┤ X ├─┤M├──╫─
          └───┘└───┘ └╥┘  ║ 
c: 2/═════════════════╩═══╩═
                      1   0 


# **Medium: Implement superdense coding for all four possible 2-bit messages ('00','01','10','11') and tabulate the decoded results.**

In [3]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

def superdense_coding(message, shots=1024):
    if message not in ("00", "01", "10", "11"):
        raise ValueError("Message must be 00, 01, 10, or 11")

    qc = QuantumCircuit(2, 2)

    # Step 1: Prepare entanglement
    qc.h(0)
    qc.cx(0, 1)

    # Step 2: Alice encodes her two-bit message
    if message[0] == "1":
        qc.x(0)
    if message[1] == "1":
        qc.z(0)

    # Step 3: Bob decodes
    qc.cx(0, 1)
    qc.h(0)

    # Step 4: Measure
    qc.measure([0, 1], [0, 1])

    result = simulator.run(qc, shots=shots).result()
    return result.get_counts()

for message in ["00", "01", "10", "11"]:
    counts = superdense_coding(message)
    decoded = max(counts, key=counts.get)
    print(f"Sent: {message} | Decoded: {decoded} | Counts: {counts}")


Sent: 00 | Decoded: 00 | Counts: {'00': 1024}
Sent: 01 | Decoded: 01 | Counts: {'01': 1024}
Sent: 10 | Decoded: 10 | Counts: {'10': 1024}
Sent: 11 | Decoded: 11 | Counts: {'11': 1024}


# **Hard: Introduce a bit-flip error on the transmitted qubit (simulate channel noise) and analyse the impact on decoding accuracy.**

In [4]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

p_flip = 0.10

# Apply X with probability p_flip, otherwise do nothing
bit_flip = pauli_error([
    ("X", p_flip),
    ("I", 1 - p_flip)
])

noise_model = NoiseModel()
noise_model.add_quantum_error(bit_flip, "id", [0])

noisy_simulator = AerSimulator(noise_model=noise_model)

def noisy_superdense_coding(message, shots=10000):
    qc = QuantumCircuit(2, 2)

    # Shared entangled pair
    qc.h(0)
    qc.cx(0, 1)

    # Encode message
    if message[0] == "1":
        qc.x(0)
    if message[1] == "1":
        qc.z(0)

    # Channel marker: the noise model acts here
    qc.id(0)

    # Decode
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])

    result = noisy_simulator.run(
        qc, shots=shots
    ).result()

    return result.get_counts()

for message in ["00", "01", "10", "11"]:
    counts = noisy_superdense_coding(message)
    correct = counts.get(message, 0)
    total = sum(counts.values())
    accuracy = 100 * correct / total

    print(
        f"Sent={message}, counts={counts}, "
        f"accuracy={accuracy:.2f}%"
    )


Sent=00, counts={'10': 1009, '00': 8991}, accuracy=89.91%
Sent=01, counts={'11': 1006, '01': 8994}, accuracy=89.94%
Sent=10, counts={'00': 958, '10': 9042}, accuracy=90.42%
Sent=11, counts={'01': 1004, '11': 8996}, accuracy=89.96%


# **Real-world: Draft a short comparison of the classical bandwidth requirement versus the superdense-coded transmission for a hypothetical 100-message communication session.**

In [5]:

# Superdense Coding vs Classical Communication
# Hypothetical session: 100 messages, 2 bits per message

num_messages = 100
bits_per_message = 2

# Classical transmission
classical_bits = num_messages * bits_per_message

# Superdense coding
qubits_transmitted = num_messages
entangled_pairs_required = num_messages
classical_bits_conveyed = qubits_transmitted * 2

# Calculate savings in transmitted carriers
carrier_reduction = (
    (classical_bits - qubits_transmitted) / classical_bits
) * 100

print("SUPERDENSE CODING vs CLASSICAL TRANSMISSION")
print("-" * 45)

print(f"Number of messages: {num_messages}")
print(f"Bits per message: {bits_per_message}")

print("\nClassical Communication:")
print(f"Classical bits transmitted: {classical_bits}")

print("\nSuperdense Coding:")
print(f"Qubits transmitted: {qubits_transmitted}")
print(f"Shared entangled pairs required: {entangled_pairs_required}")
print(f"Classical bits conveyed: {classical_bits_conveyed}")

print("\nComparison:")
print(f"Carrier-count reduction: {carrier_reduction:.1f}%")
print("Note: Entanglement distribution and preparation")
print("are additional resources for superdense coding.")


SUPERDENSE CODING vs CLASSICAL TRANSMISSION
---------------------------------------------
Number of messages: 100
Bits per message: 2

Classical Communication:
Classical bits transmitted: 200

Superdense Coding:
Qubits transmitted: 100
Shared entangled pairs required: 100
Classical bits conveyed: 200

Comparison:
Carrier-count reduction: 50.0%
Note: Entanglement distribution and preparation
are additional resources for superdense coding.


# **Challenge: Extend the protocol conceptually to transmit 2 classical bits using a noisy entangled pair with a specified fidelity, and estimate the resulting decoding error rate through simulation with a Qiskit noise model.**

In [6]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

F = 0.90
shots = 10000

# Bell-diagonal noise on Bob's qubit:
# I with probability F; X, Y, Z with probability (1-F)/3
q = (1 - F) / 3

bell_pair_error = pauli_error([
    ("I", F),
    ("X", q),
    ("Y", q),
    ("Z", q)
])

noise_model = NoiseModel()
noise_model.add_quantum_error(
    bell_pair_error, "id", [1]
)

simulator = AerSimulator(noise_model=noise_model)

def simulate_noisy_entanglement(message, shots=10000):
    qc = QuantumCircuit(2, 2)

    # Prepare ideal Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Apply the modeled entanglement noise to qubit 1
    qc.id(1)

    # Encode the classical message
    if message[0] == "1":
        qc.x(0)
    if message[1] == "1":
        qc.z(0)

    # Decode
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])

    result = simulator.run(qc, shots=shots).result()
    return result.get_counts()

total_correct = 0
total_shots = 0

for message in ["00", "01", "10", "11"]:
    counts = simulate_noisy_entanglement(message, shots)
    correct = counts.get(message, 0)
    n = sum(counts.values())

    total_correct += correct
    total_shots += n

    print(
        f"Message {message}: "
        f"accuracy = {100 * correct / n:.2f}%"
    )

accuracy = total_correct / total_shots
error_rate = 1 - accuracy

print(f"\nFidelity: {F:.2f}")
print(f"Overall accuracy: {100 * accuracy:.2f}%")
print(f"Overall error rate: {100 * error_rate:.2f}%")
print(f"Expected error rate: {100 * (1 - F):.2f}%")


Message 00: accuracy = 90.04%
Message 01: accuracy = 90.03%
Message 10: accuracy = 89.64%
Message 11: accuracy = 89.76%

Fidelity: 0.90
Overall accuracy: 89.87%
Overall error rate: 10.13%
Expected error rate: 10.00%
